# Remote Coding Agent on Google Colab

Launch the authenticated MCP server through a temporary Cloudflare Quick Tunnel. Run cells in order. Colab is ephemeral, so use short-lived, narrowly scoped credentials only.

Add `MCP_AUTH_TOKEN` to Colab Secrets. Add `GITHUB_TOKEN` only if this repository is private or you need GitHub operations. Shell commands can read credentials available to the server process.

In [ ]:
from google.colab import userdata
from getpass import getpass

def read_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = getpass(f"Enter {name} (input hidden): ")
    value = value.strip()
    if required and not value:
        raise ValueError(f"{name} is required")
    return value

MCP_AUTH_TOKEN = read_secret("MCP_AUTH_TOKEN")
GITHUB_TOKEN = read_secret("GITHUB_TOKEN", required=False)
if len(MCP_AUTH_TOKEN) < 32:
    raise ValueError("Use a high-entropy MCP_AUTH_TOKEN of at least 32 characters")
print("Secrets loaded; values are not displayed.")

In [ ]:
import os, pathlib, subprocess

REPO_DIR = pathlib.Path("/content/remote-coding-agent")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    clone_env = os.environ.copy()
    clone_env["GIT_TERMINAL_PROMPT"] = "0"
    if GITHUB_TOKEN:
        askpass = pathlib.Path("/content/.git-askpass.sh")
        askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) printf "%s" "$GITHUB_TOKEN" ;; esac\n')
        askpass.chmod(0o700)
        clone_env.update({"GITHUB_TOKEN": GITHUB_TOKEN, "GIT_ASKPASS": str(askpass)})
    try:
        subprocess.run(["git", "clone", "https://github.com/Dishant402955/remote-coding-agent.git", str(REPO_DIR)], env=clone_env, check=True)
    finally:
        if GITHUB_TOKEN:
            askpass.unlink(missing_ok=True)
print(f"Repository ready at {REPO_DIR}")

In [ ]:
%cd /content/remote-coding-agent
!pip -q install -e . httpx
print("Project dependencies installed.")

In [ ]:
import json, platform, urllib.request, subprocess, pathlib

asset_name = {"x86_64":"cloudflared-linux-amd64.deb", "amd64":"cloudflared-linux-amd64.deb", "aarch64":"cloudflared-linux-arm64.deb", "arm64":"cloudflared-linux-arm64.deb"}.get(platform.machine().lower())
if not asset_name:
    raise RuntimeError(f"Unsupported Colab architecture: {platform.machine()}")
with urllib.request.urlopen("https://api.github.com/repos/cloudflare/cloudflared/releases/latest", timeout=20) as response:
    release = json.load(response)
asset = next((item for item in release["assets"] if item["name"] == asset_name), None)
if asset is None:
    raise RuntimeError(f"Could not find {asset_name} in the latest release")
deb_path = pathlib.Path("/content/cloudflared.deb")
urllib.request.urlretrieve(asset["browser_download_url"], deb_path)
subprocess.run(["dpkg", "-i", str(deb_path)], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["cloudflared", "--version"], check=True)

In [ ]:
import os, pathlib, subprocess, time, re

workspace = pathlib.Path("/content/workspace")
workspace.mkdir(parents=True, exist_ok=True)
server_env = os.environ.copy()
server_env.update({"MCP_AUTH_TOKEN": MCP_AUTH_TOKEN, "GITHUB_TOKEN": GITHUB_TOKEN, "REMOTE_CODING_AGENT_HOST": "127.0.0.1", "REMOTE_CODING_AGENT_PORT": "8000", "REMOTE_CODING_AGENT_WORKSPACE": str(workspace)})
server = subprocess.Popen(["remote-coding-agent"], cwd=REPO_DIR, env=server_env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(2)
if server.poll() is not None:
    raise RuntimeError("MCP server exited during startup. Check configuration and dependencies.")
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tunnel_url = None
deadline = time.time() + 45
while time.time() < deadline and tunnel.poll() is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
if not tunnel_url:
    tunnel.terminate()
    server.terminate()
    raise RuntimeError("Tunnel URL not detected. Check Cloudflare availability and rerun the cell.")
print("MCP server and temporary HTTPS tunnel started.")
print("Endpoint:", tunnel_url + "/mcp")
print("Keep this runtime active. Rerunning this cell may start duplicate processes.")

In [ ]:
import httpx

response = httpx.post("http://127.0.0.1:8000/mcp", headers={"Authorization": f"Bearer {MCP_AUTH_TOKEN}", "Content-Type": "application/json", "Accept": "application/json, text/event-stream"}, json={"jsonrpc":"2.0", "id":1, "method":"initialize", "params":{"protocolVersion":"2025-03-26", "capabilities":{}, "clientInfo":{"name":"colab-smoke-check", "version":"1.0"}}}, timeout=10)
if response.status_code != 200:
    raise RuntimeError(f"Authenticated MCP initialize failed with HTTP {response.status_code}")
print("Authenticated MCP HTTP smoke check passed. This does not verify ChatGPT/client compatibility.")

## Connect your MCP client

Use the printed HTTPS endpoint ending in `/mcp` and configure `Authorization: Bearer <MCP_AUTH_TOKEN>`. Never put the token in the URL. The tunnel URL changes after runtime restart; server processes and workspace files may disappear when Colab resets.